In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import gc
import scipy.stats
sys.path.insert(0, '../tools/')

# Prevalence

In [ ]:
df_1Mb = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/del13q/depth/WGS_500k.del13q.depth.txt.gz', sep='\t')
df_age = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID_age.40709.txt', sep='\t')
df_1Mb = df_1Mb.merge(df_age, on = 'ID')

n = len(df_1Mb)
df_1Mb.loc[df_1Mb['fracInCNV']> 0.5, 'depthDev'] = df_1Mb.loc[df_1Mb['fracInCNV']> 0.5, 'relDepthInclCNV'] - 1
df_1Mb['zscore'] = df_1Mb['depthDev']/df_1Mb['SE']
# df_1Mb['zscore']=(df_1Mb['relDepthInclCNV'] - 1)/(np.sqrt(df_1Mb['EXPinclCNV'])/df_1Mb['EXPinclCNV'])

## Rescaling the zscores
df_1Mb['depthDev'] -= df_1Mb.query('age<=45 and abs(zscore)<4')['depthDev'].mean() 
df_1Mb['SE'] *= df_1Mb.query('age<=45 and abs(zscore)<4')['zscore'].std()
df_1Mb['zscore'] = df_1Mb['depthDev']/df_1Mb['SE']

# df_1Mb = df_1Mb.query('zscore < 0')
df_1Mb['z_lower'] = [x-0.5 for x in np.ceil(df_1Mb['zscore']*2)/2]
df_1Mb['z_upper'] = [x for x in np.ceil(df_1Mb['zscore']*2)/2]
df_1Mb.loc[df_1Mb['zscore']<=-5, 'z_lower'] = float('-inf')
df_1Mb.loc[df_1Mb['zscore']<=-5, 'z_upper'] = -5
df_1Mb.loc[df_1Mb['zscore']>=0, 'z_lower'] = 0
df_1Mb.loc[df_1Mb['zscore']>=0, 'z_upper'] = float('inf')

df_1Mb['age_bin'] = [f'{int(x)}-{int(x+4)}' for x in np.floor(df_1Mb['age']/5) * 5]
df_age['age_bin'] = [f'{int(x)}-{int(x+4)}' for x in np.floor(df_age['age']/5) * 5]
df_1Mb = df_1Mb.assign(del13q_depth=lambda x: (x['zscore']<-3.5).astype(int))

df_1Mb['z_lower'] = [x-0.5 for x in np.ceil(df_1Mb['zscore']*2)/2]
df_1Mb['z_upper'] = [x for x in np.ceil(df_1Mb['zscore']*2)/2]
df_1Mb.loc[df_1Mb['zscore']<=-5, 'z_lower'] = float('-inf')
df_1Mb.loc[df_1Mb['zscore']<=-5, 'z_upper'] = -5
df_1Mb.loc[df_1Mb['zscore']>=0, 'z_lower'] = 0
df_1Mb.loc[df_1Mb['zscore']>=0, 'z_upper'] = float('inf')
gc.collect()

In [ ]:
df_1Mb.query('del13q_depth==1').to_csv('~/out_dir/WGS_500k.del13q.calls.depth.txt.gz', sep='\t', index=False)

In [ ]:
age_by_zbin = df_1Mb.groupby(['z_lower', 'z_upper', 'region']).agg(
    mean_age = ('age', np.mean),
    std_age = ('age', np.std),
    count = ('ID', len)
).reset_index().assign(
    expected_counts = lambda x: np.round(n*(scipy.stats.norm().cdf(x['z_upper']) - scipy.stats.norm().cdf(x['z_lower'])), 1),
    stderr_age = lambda x: x['std_age']/np.sqrt(x['count']),
    normal_dist_true_positives=lambda x: x['count']-x['expected_counts'],
    age_dist_true_positives = lambda x: (x['count'] * (x['mean_age'] - x['mean_age'].min())/(x['mean_age'].max() - x['mean_age'].min())).round(1)
).assign(
    true_positive_fraction = lambda x: x['normal_dist_true_positives'] / x['count']
)

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
ax_age = plt.twinx(ax)
ax.bar(
    age_by_zbin.index, 
    age_by_zbin['count'],
    color='cornflowerblue',
    edgecolor='k',
)
ax.bar(
    age_by_zbin.query('z_lower<-3.5').index, 
    age_by_zbin.query('z_lower<-3.5')['normal_dist_true_positives'], 
    label='Estimated number of true positive del(13q14)',
    color='cornflowerblue',
    edgecolor='k',
    hatch='///'
)
ax.set_yscale('log')
ax_age.errorbar(
    age_by_zbin.index, 
    age_by_zbin['mean_age'], 
    yerr=age_by_zbin['stderr_age'], 
    fmt='o', 
    capsize=5,
    color='k',
    label='Mean Age'
)
ax.set_xticks(
    age_by_zbin.index, 
    [f'({x[0]},{x[1]})' for x in zip(age_by_zbin['z_lower'], age_by_zbin['z_upper'])],
    rotation=45
)
ax.set_xlabel('chr13:50-51Mb depth z-score', fontsize=16)
ax.set_ylabel('Number of individuals', fontsize=16)
ax_age.set_ylabel('Mean age (years)', fontsize=16)
ax_age.set_ylim(55.5, 63)
ax.legend(frameon=False)
plt.savefig('del13q_zscore_age_dist.pdf', bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
age_by_zbin.assign(
    zscore_enrichment=lambda x: x['count'] / x['expected_counts']
)[['z_lower', 'z_upper', 'count', 'expected_counts', 'zscore_enrichment', 'mean_age', 'stderr_age']].to_csv('13q_zscore_distribution.csv', index=False)

In [ ]:
df_snp_array = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.mCA_calls.snp_array.txt', sep='\t')
df_snp_array = df_snp_array.drop('AGE', axis = 1).merge(df_age)
prevalence_snp_array = df_snp_array \
    .query('CHR==13 and COPY_CHANGE=="loss" and END_MB>49 and START_MB<51')\
    .drop_duplicates(subset='ID') \
    .groupby('age_bin')\
    .agg(del13q_snp_array=('ID', len))\
    .reset_index()

df_hmm = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')

df_13q_calls = df_1Mb[['ID', 'del13q_depth']].merge(
    df_hmm.query('chr=="chr13" and (type=="LOSS" or type=="CN-LOH") and bpEnd>49e6 and bpStart<51e6')\
        .groupby('ID') \
        .agg(
            del13q_baf = ('type', lambda x: np.any(x=='LOSS')),
            cnloh13q = ('type', lambda x: np.any(x=='CN-LOH'))
        ),
    on='ID',
    how='outer'
) \
    .fillna(0).astype(int) \
    .assign(
        del13q_WGS = lambda x: x['del13q_depth'] | x['del13q_baf']
    )
    
prevalence = df_13q_calls.merge(df_age).groupby('age_bin').agg(
    count = ('ID', len),
    del13q_baf=('del13q_baf', sum),
    del13q_depth=('del13q_depth', sum),
    del13q_WGS=('del13q_WGS', sum),
) \
    .merge(prevalence_snp_array, on='age_bin')\
    .query('age_bin!="70-74"')


width = 0.2
fig, ax = plt.subplots(dpi=150)
ax.bar(
    prevalence.index+0*width, prevalence['del13q_snp_array']/prevalence['count']*100, 
    width=width, 
    align='edge', 
    color='slategray', 
    label='SNP-array call rate', 
    edgecolor='k'
)
ax.bar(
    prevalence.index+1*width, prevalence['del13q_baf']/prevalence['count']*100, 
    width=width,
    align='edge', 
    color='lightsteelblue', 
    label='WGS BAF call rate', 
    edgecolor='k'
)
ax.bar(
    prevalence.index+2*width, prevalence['del13q_WGS']/prevalence['count']*100, 
    width=width, 
    align='edge', 
    color='cornflowerblue', 
    label='WGS BAF+depth call rate', 
    edgecolor='k'
)

ax.set_xticks(prevalence.index+1.5*width, prevalence['age_bin'])
ax.set_xlabel('Age (years)', fontsize=18)
ax.set_ylabel('del(13q14) prevalence \n (percent)', fontsize=18)
ax.tick_params(labelsize=12)
ax.legend(frameon=False, fontsize=16)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.savefig('del13q_prevalence.pdf', bbox_inches='tight', transparent=True)

In [ ]:
df_blood_counts = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/not_restricted_to_analyzed_samples/ID.blood_counts.txt', sep='\t')
df_13q_calls = df_13q_calls.merge(df_blood_counts)

In [ ]:
df_13q_calls.groupby('del13q_WGS').agg(
    lymphocyte_count_mean = ('lymphocyte', np.mean),
    lymphocyte_count_std = ('lymphocyte', np.std),
    lymphocyte_count_q25 = ('lymphocyte', lambda x: np.percentile(x, 25)),
    lymphocyte_count_q50 = ('lymphocyte', np.median),
    lymphocyte_count_q75 = ('lymphocyte', lambda x: np.percentile(x, 75)),
).reset_index()

In [ ]:
df_plot = df_1Mb.merge(df_blood_counts).query('del13q_depth==1')
fig, ax = plt.subplots(dpi=150) 
ax.plot(-2*df_plot['depthDev'], df_plot['lymphocyte'], 'o', color='cornflowerblue', alpha=0.5, markersize=5)
ax.set_ylabel('Lymphocyte count (cells/nL)', fontsize=16)
ax.set_xlabel('Estimated del(13q14) fraction \n (per haploid genome)', fontsize=16)
ax.tick_params(labelsize=12)
ax.set_yscale('log')

# Pileups

In [ ]:
df_confident = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/del13q/WGS_500k.del13q.breakpoints.txt.gz', sep='\t')
df_confident = df_confident[df_confident['unexplainedResidual']< 9]
df_confident = df_confident[df_confident['orientation']=="INNER"]

df_confident['length'] = df_confident['bpEnd'] - df_confident['bpStart']
events_per_ind = df_confident.groupby('ID').agg(events=('ID', len)).reset_index()

# null_mu = df_confident['EXPinclCNV']
# alt_mu = ((1-(-2*df_confident['depthDev'])/2) * df_confident['EXPinclCNV']).astype(int)
# df_confident['power'] = 1-scipy.stats.poisson(mu = alt_mu).sf(scipy.stats.poisson(mu=null_mu).ppf(scipy.stats.norm.cdf(-zscore)))

In [ ]:
from tools import plot_normalized_histogram

short_deletion = df_confident[
    (np.abs((df_confident['length']/1e6-1))< .1) &
    (df_confident['depthDev'] < -0.025) &
    (df_confident['depthDev'] > -0.5) 
]

long_deletion = df_confident[
    (df_confident['length']/1e6 > 2) & 
    (df_confident['depthDev'] < -0.025) &
    (df_confident['depthDev'] > -0.5) 
]

fig, ax = plt.subplots(figsize = (8, 6), dpi=150)
bins = np.logspace(np.log10(0.05), 0, 30)


plot_normalized_histogram(-2*short_deletion['depthDev'], ax, color='#ff9408', alpha=0.7, bins=bins, label=rf'1Mb 13q deletion', normalize=False)
plot_normalized_histogram(-2*long_deletion['depthDev'], ax, color='royalblue', alpha=0.7, bins=bins, label=rf'>2Mb 13q deletion', inverted=True, normalize=False)

ax.legend(frameon=False)
ax.set_ylabel(rf'Number of 13q deletions', fontsize=14)
ax.set_xscale('log')
ax.set_xlabel('Cell fraction', fontsize=14)
ax_min, ax_max = ax.get_ylim()
ylim = np.abs([ax_min, ax_max]).max()
ax.set_ylim(-ylim, ylim)

ytickpos = np.sort(np.hstack([-np.arange(0, ylim, 5, dtype=int)[1:], np.arange(0, ylim, 5, dtype=int)]))
yticklab = np.abs(ytickpos)
ax.set_yticks(ytickpos, yticklab)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.savefig(f'del13q_fitness_effect.pdf', transparent=True, bbox_inches='tight')

# CN-LOH

In [ ]:
fig = plt.figure(figsize=(8, 8), dpi=150)
ax = fig.add_axes([0.05,0.5,0.9,0.45])
# fig.add_axes([0,0.5,0.5,1])

df_hmm = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')
data = df_hmm.query('chr=="chr13" and type=="CN-LOH"').groupby('ID').agg(
    bdev = ('bdev', np.mean),
    bpStart = ('bpStart', np.min),
    bpEnd = ('bpEnd', np.max)
).merge(df_1Mb.assign(depthDev=lambda x: x['relDepthInclCNV'] - 1).drop(['bpStart', 'bpEnd'], axis=1), on = 'ID')
data = data.query('bpStart<48e6 and bpEnd>52e6 and bdev>0.025')

ax.scatter(-2*data['depthDev'], 2*data['bdev'], marker='.', color ='b')
ax.axline((0,0), slope=1, label='monoallelic deletion', color='k')
ax.axline((0, 0), slope=1/2, label='biallelic deletion', color ='k', linestyle='--')
ax.set_ylabel('Estimated CN-LOH clonal fraction', fontsize=14)
ax.set_xlabel('-2 * (chr13:50-51Mb depth)', fontsize=14)
ax.set_xlim(-0.1, 2)
ax.set_ylim(0, 1)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.legend(frameon=False, fontsize=12)
plt.savefig('cnloh13q_and_13q_depth.pdf', transparent=True, bbox_inches='tight')

In [ ]:
high_cf_13q_CNLOH = data \
    .query('depthDev > -0.01') \
    .sort_values('bdev', ascending=False) \
    .head(10)[['ID', 'chr', 'bpStart', 'bpEnd', 'bdev']] 

high_cf_13q_CNLOH.to_csv('high_cf_13q_CNLOH_without_del.txt', sep='\t', index=False)

In [ ]:
baseline_profile = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/del13q/13qCNLOH/depth/WGS2.batch58.chr13_48_52.profile.txt.gz', sep='\t')
exclude_IDs = df_1Mb.query('del13q_depth==1').merge(
    df_hmm.query(
        'chr=="chr13" and (type=="LOSS" or type=="CN-LOH") and bpEnd>49e6 and bpStart<51e6'
    ),
    on=['ID'],
    how='outer'
)['ID']
exclude_IDs = set(exclude_IDs)

binsize=5e3

baseline_profile = baseline_profile \
    .query('inCNV==0 and EXPreads>50') \
    .query('ID not in @exclude_IDs') \
    .assign(bin = lambda x: x['bpStart'] // binsize * binsize) \
    .groupby(['ID', 'bin']) \
    .agg(
        OBSreads = ('OBSreads', sum),
        EXPreads = ('EXPreads', sum)
    ) \
    .assign(relDepth = lambda x: x['OBSreads']/x['EXPreads']) \
    .groupby('bin') \
    .agg(
        baseline = ('relDepth', np.median)
    ).reset_index().assign(
        bpStart = lambda x: x['bin'].astype(int)
    ).drop('bin', axis=1)

In [ ]:
fig, ax = plt.subplots(10, 1, figsize=(8, 8), dpi=150, sharex=True, sharey=True)
for i, ID in enumerate(high_cf_13q_CNLOH['ID']):
    profile = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/del13q/13qCNLOH/depth/{ID}.13q.depth_profile.txt', sep='\t').query('inCNV==0 and EXPreads>50')
    profile = profile.assign(bin = lambda x: x['bpStart'] // binsize * binsize).groupby('bin').agg(
        ID = ('ID', 'first'),
        OBSreads = ('OBSreads', sum),
        EXPreads = ('EXPreads', sum),
    ).reset_index().assign(
        relDepth = lambda x: x['OBSreads']/x['EXPreads'],
        bpStart = lambda x: x['bin'].astype(int)
    ).merge(baseline_profile, on='bpStart').assign(
        relDepth = lambda x: x['relDepth']/x['baseline']
    )

    ax[i].scatter(profile['bpStart']/1e6, profile['relDepth'], color='gray', alpha=1, s=2)
    ax[i].set_ylim(0.5, 1.5)
    # ax[i].set_xlim(50, 51)
    ax[i].set_xlim(48, 52)

ax[9].set_xlabel('Chromosome 13 position (Mb)', fontsize=14)
ax[4].set_ylabel('Relative depth', fontsize=14)
plt.tight_layout()
plt.savefig('high_cf_13q_CNLOH_without_del_depth_profiles.pdf', transparent=True, bbox_inches='tight')

In [ ]:
df_hmm.query('chr=="chr13" and type=="CN-LOH"').groupby('ID').agg(
    bpStart = ('bpStart', np.min),
    bpEnd = ('bpEnd', np.max)
).assign(
    overlap_50_51Mb = lambda x: (x['bpStart']<48e6) & (x['bpEnd']>52e6)
).agg(
    total_cnloh_13q = ('overlap_50_51Mb', len),
    overlap_50_51Mb = ('overlap_50_51Mb', sum)
)

In [ ]:
gwas13q = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/del13q/13qCNLOH/13qSNPs_cnloh13q.regenie.gz', sep=' ' )
fig, ax = plt.subplots(figsize=(8,6), dpi=150)
ax.scatter(gwas13q['GENPOS']/1e6, gwas13q['LOG10P'], marker='.', color='steelblue', s=1, rasterized=True)
ax.set_xlabel('Chromosome 13 position (Mb)', fontsize=14)
ax.set_ylabel(r'$-\log_{10}(P)$', fontsize=14)
# ax.set_title('Association of variants with MAC>30 on chr13 with 13q CN-LOH')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.savefig('gwas_13q_cnloh.pdf', transparent=True, bbox_inches='tight')
print(f'Min p = {10 ** -gwas13q["LOG10P"].max():.2e} among {len(gwas13q)} variants tested' )

In [ ]:
from tools import plot_normalized_histogram
df_hmm = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')
df_del13q_cnloh = df_confident.merge(
    df_hmm.query('chr=="chr13" and type=="CN-LOH"').assign(CNLOH=1).groupby('ID').agg({'CNLOH':max}).reset_index(), 
    how='left'
).replace(np.NaN, 0)


fig = plt.figure(figsize=(6, 4.5), dpi=150)

bp_hist_top = fig.add_axes([0.,0.5,1,0.5])
bp_hist_bottom = fig.add_axes([0.,0.,1,0.5], sharex=bp_hist_top)
# depmap_ax = fig.add_axes([0,0.,1,0.15], sharex=bp_hist_top)
nbins = 70
plot_normalized_histogram(
    np.concatenate([df_del13q_cnloh.query('CNLOH==0')['bpStart'], df_del13q_cnloh.query('CNLOH==0')['bpEnd']])/1e6,
    bp_hist_top,
    color = 'salmon',
    alpha = 1,
    bins = np.linspace(45, 55, nbins),
    label='CN-LOH=0',
)

plot_normalized_histogram(
    np.concatenate([df_del13q_cnloh.query('CNLOH==1')['bpStart'], df_del13q_cnloh.query('CNLOH==1')['bpEnd']])/1e6,
    bp_hist_bottom,
    color = 'lavender',
    alpha = 1,
    bins = np.linspace(45, 55, nbins),
    label='CN-LOH=1',
)
bp_hist_top.set_yticks(np.arange(0, 0.351, 0.1))
bp_hist_bottom.set_yticks(np.arange(0.1, 0.351, 0.1))
bp_hist_top.set_ylim(0, 0.35)
bp_hist_bottom.set_ylim(0, 0.35)
bp_hist_bottom.invert_yaxis()

for spine in ['bottom', 'top', 'right']:
    bp_hist_bottom.spines[spine].set_visible(False)
    bp_hist_top.spines[spine].set_visible(False)
bp_hist_bottom.set_xlabel('Chromosome 13 position (Mb)', fontsize=18)
bp_hist_top.text(-0.1, 0, 'Distribution of del(13q14) breakpoints', transform=bp_hist_top.transAxes, ha='left', va='center', fontsize=18, rotation=90)
bp_hist_top.text(0.1, 0.75, 'without 13q CN-LOH', transform=bp_hist_top.transAxes, ha='left', va='center', fontsize=16)
bp_hist_bottom.text(0.1, 0.25, 'with 13q CN-LOH', transform=bp_hist_bottom.transAxes, ha='left', va='center', fontsize=16)
bp_hist_top.tick_params(labelsize=12)
bp_hist_bottom.tick_params(labelsize=12)


gtf = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/hg38.refGene.txt', sep='\t')
depmap = pd.read_csv('/mnt/project/lohdata/david/resources/Gene.BloodDepMapScores.txt', sep='\t')
depmap['quantile'] = pd.qcut(depmap['lymphoid_depmap'], 10, labels=False)
gtf = gtf.query('chr=="chr13" and start<55e6 and end>45e6').groupby('gene').agg(
    start = ('start', np.min),
    end = ('end', np.max)
)

cancer_genes = pd.read_csv('/mnt/project/lohdata/david/resources/cancerGeneList.tsv', sep='\t')
cancer_genes = cancer_genes.assign(
    gene = lambda x: x['Hugo Symbol'].str.upper(),
    tsg = lambda x: x['Is Tumor Suppressor Gene'].fillna('No').replace({'Yes': True, 'No': False}),
    oncogene = lambda x: x['Is Oncogene'].fillna('No').replace({'Yes': True, 'No': False}),
)[['gene', 'tsg', 'oncogene']]
gtf = gtf.merge(cancer_genes, on='gene', how = 'left').fillna(False)
gtf = gtf.merge(depmap, on='gene', how='left')

for idx, row in gtf.query('quantile<1').iterrows():
    ax = bp_hist_bottom if row['gene'] == 'TPT1' else bp_hist_top
    xpos = (row['start']+row['end'])/2/1e6

    ax.text(
        xpos, 
        0.15, 
        rf'$\it{{{row["gene"]}}}$', 
        rotation=90, 
        ha='right', 
        va='bottom' if row['gene'] == 'TPT1' else 'top', 
        fontsize=12,
        color='gray'
    )
    ax.plot([xpos, xpos], [0, 0.15], color='gray', linewidth=1.5)
    # ax.axvline((row['start']+row['end'])/2/1e6, color='k', ymin=0.1)
plt.savefig('cnloh13q_with_del13q.pdf', transparent=True, bbox_inches='tight')


In [ ]:
import matplotlib.cm as cm

chrom=13
fig = plt.figure(figsize=(4, 4.5), dpi=300)
ax1 = fig.add_axes([0.1, 0.1, 0.9, 0.8])
axins = fig.add_axes([0.1, 0.98, 0.9, 0.02])

df_plot = df_confident
y_coord = np.argsort(np.argsort(df_plot['bpStart'] - df_plot['length']))
colors = (-2*df_plot['depthDev'])
norm = plt.Normalize(0, 0.25)
cmap = cm.plasma
ax1.hlines(
    xmin=df_plot['bpStart']/1e6, 
    xmax=df_plot['bpEnd']/1e6, 
    y=y_coord, 
    color=cmap(norm(colors)),
    alpha=0.8,
    cmap='bwr', 
    linewidth=0.5
)
ax1.set_ylabel('13q DEL event number',fontsize=16)
# ax1.set_xticks([])
ax1.spines['left'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.spines['bottom'].set_visible(True)
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=axins, label='Cell Fraction', orientation='horizontal')
ax1.set_xlabel(f'Chromosome {chrom} position (Mb)', fontsize=16)
ax1.tick_params(labelsize=12)

plt.savefig(f'del13q_pileup.pdf', transparent=True, bbox_inches='tight')
plt.show()

## Hi-C

In [ ]:
from tools import plot_gtf
hiC = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/del13q/HiC/Vilarrasa-Blasi.NBC_123.c13.45_55.txt', sep='\t', header=None)
bins = np.arange(45000000,55000000+1,20000)
hiC.index = bins
hiC.columns = bins

gtf = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/hg38.refGene.txt', sep='\t')


fig = plt.figure(figsize=(8, 8), dpi=150)
ax2 = fig.add_axes([0.8, 0.2, 0.2, 0.6])
ax3 = fig.add_axes([0.2, 0.8, 0.6, 0.2])
ax1 = fig.add_axes([0.2, 0.2, 0.6, 0.6])
ax4 = fig.add_axes([0.2, 0, 0.6, 0.16])
ax5 = fig.add_axes([0, 0.2, 0.16, 0.6])

ax2.hist(
    np.hstack([df_confident['bpStart'], df_confident['bpEnd']]),
    bins = bins, 
    alpha=0.3, 
    orientation='horizontal',
    color='k'
)
ax3.hist(
    np.hstack([df_confident['bpStart'], df_confident['bpEnd']]),
    bins = bins, 
    alpha=0.3, 
    color='k'
)
ax2.set_ylim((bins.min(), bins.max()))
ax3.set_xlim((bins.min(), bins.max()))


plot_gtf(gtf, f'chr{chrom}', bins.min(), bins.max(), ax4, highlight_genes={'DNMT3A', 'DLEU2', 'TET2', 'DLEU1', 'DLEU7'}, names=True, fontsize=8)
plot_gtf(gtf, f'chr{chrom}', bins.min(), bins.max(), ax5, highlight_genes={'DNMT3A', 'DLEU2', 'TET2', 'DLEU1', 'DLEU7'}, transpose=True)
ax1.imshow(hiC.loc[bins, bins], vmax=50, cmap='Reds')
# ax1.set_xticks(np.arange(0, len(hiC)+1, 50), bins[::50]/1e6)
# ax1.set_yticks(np.arange(0, len(hiC)+1, 50), bins[::50]/1e6)


ax2.set_ylim(bins.min(), bins.max())
ax3.set_xlim(bins.min(), bins.max())

ax4.set_xlim(bins.min()/1e6, bins.max()/1e6)
ax5.set_ylim(bins.min()/1e6, bins.max()/1e6)

ax4.set_ylim(-10, 10)
ax5.set_xlim(-10, 10)
ax2.invert_yaxis()
ax5.invert_yaxis()
ax4.set_xlabel(f'Chromosome {chrom} position (Mb)', fontsize=24)
ax5.set_ylabel(f'Chromosome {chrom} position (Mb)', fontsize=24)

for ax in ax1, ax2, ax3, ax4, ax5:
    if ax != ax1:
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])


ax1.set_xticks(np.arange(len(bins), step=50), bins[::50]//1000000)
ax1.set_yticks(np.arange(len(bins), step=50), bins[::50]//1000000)
plt.savefig(f'del13q_HiC.pdf', transparent=True, bbox_inches='tight')
plt.show()

# CLL Survival

In [ ]:
import time
def date2years(x):
    return time.mktime(time.strptime(x, '%Y-%m-%d'))/86400/365.25

cancer_registry = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.histology.behavior.sample_date.cancer_date.txt', sep ='\t')
death_registry = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.death_data.txt', sep='\t')
collection_date =  pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.sample_date.txt', sep='\t')
prevalent_cancer_ID = set(cancer_registry.query('histology>=9590 and cancer_date < collection_date')['ID'])

cll = cancer_registry.query('histology==9823 and behavior==3')[['ID','cancer_date']].sort_values('cancer_date')
cll = cll[~cll['ID'].duplicated(keep='last')]
end_date = "2018-01-01"

df_age = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID_age.40709.txt', sep='\t')
df_sex = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.genetic_sex.txt', sep='\t')

df_hmm = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')
df_hmm = df_hmm\
    .query('chr=="chr13" and type == "CN-LOH"')\
    .groupby('ID')\
    .agg(cf = ('cf', np.max))\
    .reset_index()\
    .assign(type='CN-LOH')

df = df_confident \
    .assign(cf = lambda x: -2*x['depthDev']) \
    .groupby('ID') \
    .agg(cf = ('cf', np.max)) \
    .reset_index() 

# df = df_1Mb.assign(cf=lambda x: -2*x['depthDev']).query('del13q_depth==1')[['ID', 'cf']]

survival = collection_date.merge(df[['ID', 'cf']], on = 'ID', how = 'left')\
    .merge(cll, on='ID', how = 'left')\
    .merge(death_registry, on = 'ID', how = 'left')
survival = survival[(survival['collection_date']> '2006')]
survival['censor_date'] = np.minimum(survival['death_date'].replace(np.NaN, end_date).to_numpy(), end_date)

survival['cancer'] = (survival['cancer_date'] <= survival['censor_date']).astype(int)
survival['death'] = (survival['death_date'] <= survival['censor_date']).astype(int)


survival['del13q'] = (1 - survival['cf'].isna().astype(int))
df_end_date = np.array([date2years(x) for x in np.min(survival[['cancer_date', 'censor_date']].replace(np.NaN, end_date), axis = 1)])
df_start_date = np.array([date2years(x) for x in survival['collection_date']])
survival['duration'] =  df_end_date - df_start_date
survival = survival[[x not in prevalent_cancer_ID for x in survival['ID']]]
survival = survival[['ID', 'del13q', 'cf', 'cancer', 'death', 'duration']]
survival = survival.merge(df_sex, on='ID').merge(df_age, on='ID')
survival = survival\
    .rename({'cf':'cf_DEL'}, axis = 1)\
    .merge(df_hmm[['ID', 'cf', 'type']], on = 'ID', how='left')\
    .rename({'cf':'cf_CNLOH', 'type':'CNLOH'}, axis = 1)\
    .assign(CNLOH=lambda x: (~x['CNLOH'].isna()).astype(int))

# filter for EUR ancestry
df_ancestry = pd.read_csv('/mnt/project/lohdata/ronen/resources/genetic_ancestry.txt', sep=' ')
survival = df_ancestry.rename(
    {'UKBID':'ID', 'ETH_ga':'ancestry'}, axis=1
).merge(survival, how = 'inner').query('ancestry=="EUR"')
gc.collect()

In [ ]:
pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/del13q/CLL_progression/CLL_pheno.txt', sep ='\t') \
    .query('del13q_tri12==1')[['FID', 'IID', 'cf']] \
    .merge(survival[['ID', 'cancer', 'duration']], left_on='FID', right_on='ID') \
    .assign(
        cf = lambda x: np.round(x['cf'], 4),
        duration = lambda x: np.round(x['duration'], 4)
    ) \
    .drop('ID', axis = 1) \
    .to_csv('~/out_dir/CLL_progression_pheno.txt', sep='\t', index=None)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
X = survival[['genetic_sex', 'age']]
y = survival['del13q']
model.fit(X, y)
survival['propensity_score'] = model.predict_proba(X)[:,1]

np.random.seed(123)
del13q = survival[survival['del13q'] == 1]
control_ids = set(df_13q_calls.query('del13q_WGS == 0 and cnloh13q==0')['ID'])
control = survival.query('ID in @control_ids')

idxs = set() 
for i, treated_row in del13q.iterrows():
    idxs = idxs.union(set(control[(control['propensity_score'] - treated_row['propensity_score']).abs() == 0].sample(5, replace=False).index))
matched_control = control.loc[list(idxs)]

In [ ]:
!pip install lifelines -q
from lifelines import KaplanMeierFitter
def kaplan_meier_curve(df, ax = None, max_t=10, label=None):
    kmf = KaplanMeierFitter()
    data = np.array(
        [
            [x, y] if x<max_t else [10, 0] 
            for x, y in zip(df['duration'], df['cancer'])
        ]
    )
    if ax is None:
        ax = plt.gca()
    kmf.fit(
        data[:, 0], 
        data[:, 1], 
        label=label
    )
    kmf.plot(ci_show=True, linewidth=2, alpha=0.5, ax=ax)
    estimate = kmf.survival_function_.iloc[-1].item()
    ci = kmf.confidence_interval_.iloc[-1].to_numpy()
    print(f"SF at time {kmf.timeline[-1]} for {label}: {np.round(estimate,4)} {np.round(ci[0],4), np.round(ci[1],4)}" )
    return kmf

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, dpi=150, sharey=True, figsize=(8, 4))

print('del13q')
kaplan_meier_curve(matched_control, ax1, label='Controls')
df_plot = survival.query("del13q==1 and cf_DEL<0.05 and CNLOH==0")
kaplan_meier_curve(df_plot, ax1, label=f'<0.05 cf (n={len(df_plot)})')
df_plot = survival.query("del13q==1 and cf_DEL>=0.05 and cf_DEL<0.1 and CNLOH==0")
kaplan_meier_curve(df_plot, ax1, label=f'0.05-0.1 cf (n={len(df_plot)})')
df_plot = survival.query("del13q==1 and cf_DEL>=0.1 and CNLOH==0")
kaplan_meier_curve(df_plot, ax1, label=f'>0.1 cf (n={len(df_plot)})')
print()

print('13q CN-LOH')
kaplan_meier_curve(matched_control, ax2, label='Controls')
df_plot = survival.query("cf_CNLOH<0.05 and CNLOH==1")
kaplan_meier_curve(df_plot, ax2, label=f'<0.05 cf (n={len(df_plot)})')
df_plot = survival.query("cf_CNLOH>=0.05 and cf_CNLOH<0.1 and CNLOH==1")
kaplan_meier_curve(df_plot, ax2, label=f'0.05-0.1 cf (n={len(df_plot)})')
df_plot = survival.query("cf_CNLOH>=0.1 and CNLOH==1")
kaplan_meier_curve(df_plot, ax2, label=f'>0.1 cf (n={len(df_plot)})')

ax1.legend(frameon=False)
ax2.legend(frameon=False)
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.tick_params(left=False, labelleft=False)
plt.subplots_adjust(wspace=0.1) 

ax1.set_xlabel('Years of follow-up', fontsize=14)
ax2.set_xlabel('Years of follow-up', fontsize=14)
ax1.set_ylabel('CLL free survival', fontsize=14)
ax1.set_xticks(np.arange(0, 10+1, 1))
ax2.set_xticks(np.arange(0, 10+1, 1))
ax1.grid(visible=True, linestyle='--', alpha=0.5, axis='y')
ax2.grid(visible=True, linestyle='--', alpha=0.5, axis='y')
ax1.spines['left'].set_visible(False)
ax1.set_title('13q DEL', fontsize=16)
ax2.set_title('13q CN-LOH', fontsize=16)
plt.savefig('CLL_survival.pdf', transparent=True, bbox_inches='tight')
plt.show()

# CLL GWAS

In [ ]:
from tools import manhattan_plot, downsample_gwas

fig, ax = plt.subplots(2, 2, dpi=150, figsize=(10, 10))
ax = ax.flatten()
GWAS_DIR='/mnt/project/lohdata/david/mCAs_WGS/del13q/GWAS'
for i,pheno in enumerate(['C911', 'del13q', 'cnloh13q', 'tri12']):
    df = pd.read_csv(f'{GWAS_DIR}/{pheno}.CLL.GWAS_catalog.index_var.txt', sep='\t')
    ax[i].errorbar(
        np.exp(df['BETA']), 
        df['CLL_OR'], 
        marker='.', 
        linestyle='None', 
        yerr=(df['CLL_OR'] - df['CLL_OR_LOWER'], df['CLL_OR_UPPER'] -df ['CLL_OR']),
        xerr=(
            np.exp(df['BETA']) - np.exp(df['BETA'] - 1.96*df['SE']) ,
            np.exp(df['BETA'] + 1.96*df['SE']) - np.exp(df['BETA'])
        )
    )
    ax[i].axline((1,1), slope=1)
    ax[i].axhline(1, linestyle='--', color='grey')
    ax[i].axvline(1, linestyle='--', color='grey')
    ax[i].set_xlabel(f'{pheno} GWAS odds ratio', fontsize=14)
    ax[i].set_ylabel('Law et al. CLL GWAS odds ratio', fontsize=14)
for axis,label in zip(ax, ['a', 'b', 'c', 'd']):
    axis.text(-0.1, 1.1, label, transform=axis.transAxes, fontsize=24, va='top', ha='left')
plt.tight_layout()
plt.savefig('CLL_GWAS_beta_beta.pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(8, 8))
GWAS_DIR='/mnt/project/lohdata/david/mCAs_WGS/del13q/GWAS'
df_C911 = pd.read_csv(f'{GWAS_DIR}/C911.CLL.GWAS_catalog.index_var.txt', sep='\t')
df_mCA = pd.read_csv(f'{GWAS_DIR}/del13q_tri12.CLL.GWAS_catalog.index_var.txt', sep='\t')
ax.errorbar(
    y=np.exp(df_C911['BETA']), 
    x=np.exp(df_mCA['BETA']),
    marker='.', 
    linestyle='None', 
    yerr=(df_C911['CLL_OR'] - df_C911['CLL_OR_LOWER'], df_C911['CLL_OR_UPPER'] - df_C911['CLL_OR']),
    xerr=(df_mCA['CLL_OR'] - df_mCA['CLL_OR_LOWER'], df_mCA['CLL_OR_UPPER'] - df_mCA['CLL_OR']),
)

ax.set_xlim((0.7, 1.7))
ax.set_ylim((0.7, 1.7))

X = np.vstack([np.repeat(1, len(df_mCA)), np.exp(df_mCA['BETA'])]).T
y = np.exp(df_C911['BETA'])
b, m = np.linalg.solve(X.T @ X, X.T @ y)
ax.plot([0, 3], [b, b+3*m], color='k')

ax.axhline(1, linestyle='--', color='grey')
ax.axvline(1, linestyle='--', color='grey')
ax.set_xlabel(f'del(13q)+tri(12) GWAS odds ratio', fontsize=14)
ax.set_ylabel('C911 GWAS odds ratio', fontsize=14)

In [ ]:
fig,ax = plt.subplots(2, 1, dpi=300, figsize=(8,5))
ax = ax.flatten()


loci = {
    'del13q_tri12': ['TERC', 'PRF1', 'ETS1'],
    'C911': ['HHEX']
}

alignments = {
    'del13q_tri12': ['center', 'right', 'left'],
    'C911': ['left']
}

for i, pheno in enumerate(['del13q_tri12', 'C911']):
    print(pheno)
    df = pd.read_csv(f'{GWAS_DIR}/{pheno}.GWAS.regenie.downsampled.txt.gz', sep='\t')
    pos = manhattan_plot(df, ax[i])

    index = df.query('NOVEL').index

    ax[i].scatter(np.array(pos)[index], df.loc[index, 'LOG10P'], color='r', marker='.', s=5)

    ys = df.query('NOVEL and INDEX')['LOG10P'].to_numpy()
    xs = np.array(pos)[df.query('NOVEL and INDEX').index]
    for text, ha, x, y in zip(loci[pheno], alignments[pheno], xs, ys):
        ax[i].annotate(rf'$\it{{{text}}}$', xy=(x, y), xytext=(x,y+5), ha=ha, va='bottom', fontsize=10, arrowprops=dict(arrowstyle='-', lw=1))
    ax[i].set_ylim(0, 30)
    ax[i].axhline(-np.log10(5e-8), color='k', linestyle='--', linewidth=1)
    ax[i].spines['right'].set_visible(False)
    ax[i].spines['top'].set_visible(False)
    ax[i].spines['bottom'].set_visible(False)
    ax[i].set_ylabel(r'$-log_{10}(p)$', fontsize=14)
    
ax[1].invert_yaxis()
ax[1].xaxis.tick_top()
ax[1].xaxis.set_ticklabels([])

ax[0].text(0.5, 1, 'del(13q14)+tri(12) GWAS', transform=ax[0].transAxes, ha='center', va='bottom', fontsize=14)
ax[1].text(0.5, 0, 'CLL GWAS', transform=ax[1].transAxes, ha='center', va='top', fontsize=14)
plt.savefig('del13q_tri12_vs_C911_miami.pdf', transparent=True, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
df_CLL_index = pd.read_csv(f'{GWAS_DIR}/C911.CLL.GWAS_catalog.top_pval.txt', sep='\t')
df_del13q_index = pd.read_csv(f'{GWAS_DIR}/del13q_tri12.CLL.GWAS_catalog.top_pval.txt', sep='\t')
df_index = pd.merge(df_CLL_index, df_del13q_index, on = ['CHROM', 'POS'])
fig,ax = plt.subplots(dpi=150, figsize=(5, 5))
ax.fill_betweenx([-2, 100], -np.log10(5e-8), 100, alpha=0.5, color='#66c2a5', label='Significant in del(13q14) GWAS')
ax.fill_between([-2, 100], -np.log10(5e-8), 100, alpha=0.3, color='#fc8d62', label='Significant in CLL GWAS')
ax.axhline(-np.log10(5e-8), color='k', linestyle='--')
ax.axvline(-np.log10(5e-8), color='k', linestyle='--')
ax.axline((0,0), slope=1, color='k')
ax.scatter(df_index['LOG10P_BEST_y'], df_index['LOG10P_BEST_x'], marker='o', linestyle='None', edgecolors='k')
ax.set_xlabel(r'$-\log_{10}(p)$ del(13q14)+tri(12) GWAS', fontsize=14)
ax.set_ylabel(r'$-\log_{10}(p)$ CLL GWAS', fontsize=14)
ax.set_xlim(-1, 30)
ax.set_ylim(-1, 30)
ax.legend()
plt.savefig('del13q_tri12_vs_C911_sig_power.pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
res = []
for chrom in range(1, 22+1):
    df = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/del13q/CLL_progression/chr{chrom}.bt.survival_cancer.regenie.gz', sep=' ')    
    df = downsample_gwas(df)
    res.append(df)
    print(chrom)
df = pd.concat(res)
del(res)
gc.collect()

fig, ax = plt.subplots(dpi=150, figsize=(8,5))
manhattan_plot(df, ax)
ax.axhline(-np.log10(5e-8), color='r')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_ylabel(r'$-log_{10}(p)$', fontsize=14)
ax.set_title('Progression to CLL from mCA precursor', fontsize=16)
ax.set_ylim(0, 10)
